In [ ]:
import pandas as pd

In [ ]:
latest_ods = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses.csv")

In [ ]:
one_with_missing_2023 = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses-0106.csv")

In [ ]:
older_ods = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-01-overdoses_old.csv")

In [ ]:
latest_ods['DeathDate']  = pd.to_datetime(latest_ods['DeathDate'])

In [ ]:
latest_ods_2023 = latest_ods[(latest_ods['DeathDate'] > '2023-05-31') & (latest_ods['DeathDate'] < '2023-12-31')]

In [ ]:
older_ods['DeathDate']  = pd.to_datetime(older_ods['DeathDate'])

In [ ]:
older_ods_2023 = older_ods[(older_ods['DeathDate'] > '2023-05-31') & (older_ods['DeathDate'] < '2023-12-31')]

In [ ]:
older_ods_2023['CaseNumber']

In [ ]:
merged = older_ods_2023.merge(latest_ods_2023, on='CaseNumber', how='outer', indicator=True)

missing_in_latest_ods_2023 = merged[merged['_merge'] == 'left_only']
missing_in_older_ods_2023 = merged[merged['_merge'] == 'right_only']

In [ ]:
missing_in_latest_ods_2023['source_file_x'].value_counts()

In [ ]:
def overdoses_per_year(df, deathdate_col="DeathDate"):
    # Parse DeathDate safely
    dt = pd.to_datetime(df[deathdate_col], errors="coerce")
    # Count rows per year
    return dt.dt.year.value_counts(dropna=True).sort_index()

# Replace df1 and df2 with your actual dataframe names
counts_1 = overdoses_per_year(latest_ods)
counts_2 = overdoses_per_year(older_ods)
count_3 = overdoses_per_year(one_with_missing_2023)

# Combine into one table
overdose_table = pd.concat([counts_1, counts_2, count_3], axis=1).fillna(0).astype(int)
overdose_table.columns = ["Latest File (0107)", "File before 2023 data missing", "Previous file"]

# Make it a nice table with Year as a column
overdose_table = overdose_table.reset_index().rename(columns={"index": "Year"})

overdose_table

In [ ]:
import pandas as pd

def overdoses_per_year_flag(df, flag_col, deathdate_col="DeathDate"):
    """
    Counts overdoses per year where df[flag_col] indicates presence of a drug.
    Works with 1/0, True/False, Y/N, Yes/No, etc.
    """
    # Parse DeathDate safely
    dt = pd.to_datetime(df[deathdate_col], errors="coerce")

    # Create a robust "is flagged" mask
    flagged = df[flag_col].astype(str).str.strip().str.lower().isin(
        ["1", "true", "t", "y", "yes"]
    )

    # Filter dates to flagged rows only
    dt_flagged = dt[flagged]

    # Count rows per year
    return dt_flagged.dt.year.value_counts(dropna=True).sort_index()


def make_flag_table(flag_col, deathdate_col="DeathDate"):
    counts_1 = overdoses_per_year_flag(latest_ods, flag_col, deathdate_col)
    counts_2 = overdoses_per_year_flag(older_ods, flag_col, deathdate_col)
    counts_3 = overdoses_per_year_flag(one_with_missing_2023, flag_col, deathdate_col)

    overdose_table = pd.concat([counts_1, counts_2, counts_3], axis=1).fillna(0).astype(int)
    overdose_table.columns = ["Latest File (0107)", "File before 2023 data missing", "Previous file"]

    overdose_table = overdose_table.reset_index().rename(columns={"index": "Year"})
    return overdose_table


# --- Create the two tables ---
fentanyl_table = make_flag_table("Fentanyl")
meth_table = make_flag_table("Methamphetamine")

fentanyl_table

In [ ]:
fentanyl_table

In [ ]:
latest_ods['Race'] = latest_ods['Race'].replace({"middleeasternornorthafrican": "MIDDLE EASTERN", "cardiovasculardiseasenull": np.nan, "ck": np.nan, "k": np.nan})

In [ ]:
latest_ods.to_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses-cleaned-1219.csv")

In [ ]:
old_latest.columns

In [ ]:
KEYS = ["CaseNumber"]

In [ ]:
removed = (
    old_latest
    .merge(
        latest_ods[KEYS],
        on=KEYS,
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

print(len(removed))

In [ ]:
removed['CaseNumber']

In [ ]:
removed

In [ ]:
cases_24_new[cases_24_new['CaseNum'] == '2024-00046']['InjuryDesc']

In [ ]:
cases_24[cases_24['CaseNum'] == '2024-00046']['InjuryDesc']

In [ ]:
cases_24['InjuryDesc']